# Evered 2023 parallel CZ pulse reproduction

This notebook is a working pad for reproducing the pulse-optimization part of Evered et al., Nature 622, 268-272 (2023), DOI `10.1038/s41586-023-06481-y`.

Scope used here:

- Start from the Methods fixed-amplitude time-optimal ansatz `phi(t) = A cos(omega t - phi0) + delta0 t`.
- Verify the paper parameters against the repository's ideal CZ model.
- Inspect the two-photon/dark-state conventions that matter before adding scattering and laser-noise layers.
- Read the existing `evered2023_parallel_cz` GRAPE scan artifact and keep a small optional smoke optimization cell for interactive experimentation.

The paper platform is `87Rb`; this repository line currently uses the existing neutral-Yb model/optimizer infrastructure as a dimensionless validation harness for the analytic pulse family.

In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from pprint import pprint

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

os.environ.setdefault("MPLCONFIGDIR", str(ROOT / ".cache" / "matplotlib"))

from neutral_yb.config.artifact_paths import evered2023_parallel_cz_dir
from neutral_yb.config.species import idealised_yb171
from neutral_yb.models.evered2023_parallel_cz import (
    Evered2023DarkStateConfig,
    Evered2023ParallelCZCalibration,
    Evered2023TimeOptimalPulse,
    build_evered2023_ideal_global_cz_model,
    build_evered2023_two_photon_detuning_model,
)
from neutral_yb.optimization.evered2023_parameterized_grape import (
    Evered2023ParameterizedGRAPEConfig,
    Evered2023ParameterizedGRAPEOptimizer,
    Evered2023TwoPhotonDetuningGRAPEOptimizer,
)
from neutral_yb.optimization.global_phase_grape import (
    GlobalPhaseOptimizationConfig,
    PaperGlobalPhaseOptimizer,
)

plt.rcParams.update({"figure.figsize": (8, 4.5), "axes.grid": True})
ROOT

## 1. Paper parameters encoded in the repository

The fixed-amplitude Methods ansatz is

`phi(t) = A cos(omega t - phi0) + delta0 t`,

with `A/2pi = 0.1122`, `omega/Omega = 1.0431`, `phi0 = -0.7318`, `delta0/Omega = 0`, and `Omega T / 2pi = 1.215`. At the reported `Omega/2pi = 4.6 MHz`, this is about `264 ns`.

In [ ]:
pulse = Evered2023TimeOptimalPulse()
calibration = Evered2023ParallelCZCalibration()

pprint(pulse.to_json())
print()
pprint(calibration.to_json())

In [ ]:
num_points = 600
times = np.linspace(0.0, pulse.dimensionless_duration, num_points)
phase = pulse.phase(times)
detuning = pulse.two_photon_detuning(times)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(times / (2.0 * np.pi), phase, color="#345995", linewidth=2)
axes[0].set_xlabel("Omega t / 2pi")
axes[0].set_ylabel("phi(t) [rad]")
axes[0].set_title("Methods Eq. (1) phase")

axes[1].plot(times / (2.0 * np.pi), detuning, color="#c44536", linewidth=2)
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].set_xlabel("Omega t / 2pi")
axes[1].set_ylabel("delta(t) / Omega")
axes[1].set_title("Detuning gauge: delta(t) = -d phi / dt")
fig.tight_layout()

## 2. Ideal exact-gate check

This is the fastest sanity check: sample the analytic phase profile, propagate it through the repository's ideal 4D global-CZ reference, then optimize only the final local/global phase parameter `theta` used by the phase-gate fidelity objective.

In [ ]:
model_4d = build_evered2023_ideal_global_cz_model(species=idealised_yb171())
slot_optimizer = PaperGlobalPhaseOptimizer(
    model=model_4d,
    config=GlobalPhaseOptimizationConfig(
        num_tslots=220,
        evo_time=pulse.dimensionless_duration,
        max_iter=1,
    ),
)

sampled_phases = pulse.sampled_phases(slot_optimizer.config.num_tslots)
state = slot_optimizer.final_state(sampled_phases)
theta, fidelity = model_4d.optimize_theta_for_state(state)

print(f"theta = {theta:.9f} rad")
print(f"ideal 4D fidelity = {fidelity:.12f}")
print("final state amplitudes:")
for label, amplitude in zip(model_4d.basis_labels(), state):
    print(f"  |{label}>: {amplitude.real:+.8f}{amplitude.imag:+.8f}j")

## 3. Dark-state sign convention

Evered et al. choose the signs of the intermediate detuning and initial two-photon detuning to reduce intermediate-state scattering. The helper below exposes the Methods Eq. (2) single-atom ladder Hamiltonian and leading-order dark/bright/eigenvector diagnostics.

In [ ]:
dark_cfg = Evered2023DarkStateConfig(
    omega_blue=calibration.blue_rabi_hz / calibration.omega_over_2pi_hz,
    omega_red=calibration.red_rabi_hz / calibration.omega_over_2pi_hz,
    intermediate_detuning=calibration.intermediate_detuning_hz / calibration.omega_over_2pi_hz,
    two_photon_detuning=float(detuning[0]),
)

print("Single-atom Eq. (2) Hamiltonian, in units of effective Omega:")
print(np.asarray(dark_cfg.hamiltonian().full()).real)
print()

vectors = dark_cfg.dark_bright_eigenvectors_leading_order()
for name, vector in vectors.items():
    print(f"{name}: norm={np.linalg.norm(vector):.6f}, intermediate amplitude={vector[1]:+.6e}")

labels = ["D", "B", "E"]
intermediate_weights = [abs(vectors[label][1]) ** 2 for label in labels]
plt.bar(labels, intermediate_weights, color=["#2a9d8f", "#e76f51", "#6d597a"])
plt.ylabel("leading-order |e> weight")
plt.title("Intermediate-state content in dressed-state diagnostics");

## 4. Existing GRAPE scan artifact

The repository already contains a parameterized-GRAPE scan under `artifacts/evered2023_parallel_cz`. The artifact records whether random-restart optimization can recover high-fidelity members of the fixed-amplitude analytic family. It is a GRAPE validation artifact, not a full paper-level error-budget reproduction.

In [ ]:
artifact_path = evered2023_parallel_cz_dir(ROOT) / "evered2023_parameterized_grape_scan.json"
if not artifact_path.exists():
    raise FileNotFoundError(
        f"Missing {artifact_path}. Run experiments/reproduce_evered2023_parallel_cz_gate.py first."
    )

summary = json.loads(artifact_path.read_text(encoding="utf-8"))
scan = summary["scan"]
best = scan["best_fidelity_result"]

print(summary["purpose"])
print(f"model: {summary['grape_setup']['model']}")
print(f"target fidelity: {scan['target_fidelity']:.8f}")
print("best fidelity result:")
pprint({
    "OmegaT/2pi": best["omega_t_over_2pi"],
    "fidelity": best["fidelity"],
    "A/2pi": best["amplitude_phase_modulation_over_2pi"],
    "omega/Omega": best["phase_rate_over_omega"],
    "phi0": best["phase_offset_rad"],
    "delta0/Omega": best["static_detuning_over_omega"],
})

In [ ]:
durations = np.asarray(scan["durations_omega_t_over_2pi"], dtype=float)
fidelities = np.asarray(scan["fidelities"], dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

axes[0].plot(durations, fidelities, marker="o", color="#345995", linewidth=2)
axes[0].axhline(scan["target_fidelity"], color="#c44536", linestyle="--", label="target")
axes[0].axvline(pulse.omega_t_over_2pi, color="black", linestyle=":", label="paper 1.215")
axes[0].set_xlabel("Omega T / 2pi")
axes[0].set_ylabel("fidelity")
axes[0].set_title("Parameterized GRAPE scan")
axes[0].legend()

labels = ["A/2pi", "omega/Omega", "phi0", "delta0/Omega", "OmegaT/2pi"]
paper_values = np.array([
    pulse.amplitude_phase_modulation / (2.0 * np.pi),
    pulse.phase_rate,
    pulse.phase_offset,
    pulse.static_detuning,
    pulse.omega_t_over_2pi,
])
best_values = np.array([
    best["amplitude_phase_modulation_over_2pi"],
    best["phase_rate_over_omega"],
    best["phase_offset_rad"],
    best["static_detuning_over_omega"],
    best["omega_t_over_2pi"],
])
axes[1].bar(labels, best_values - paper_values, color="#6d597a")
axes[1].axhline(0.0, color="black", linewidth=1)
axes[1].tick_params(axis="x", rotation=25)
axes[1].set_ylabel("best scan - paper")
axes[1].set_title("Parameter displacement")

fig.tight_layout()

## 5. Optional interactive smoke optimization

Set `RUN_SMOKE_GRAPE = True` to run a tiny random-restart optimization in the ideal model. This is deliberately small so the notebook stays responsive; the full scan remains the experiment script.

In [ ]:
RUN_SMOKE_GRAPE = False

if RUN_SMOKE_GRAPE:
    smoke_optimizer = Evered2023ParameterizedGRAPEOptimizer(
        model=model_4d,
        omega_t_over_2pi=pulse.omega_t_over_2pi,
        config=Evered2023ParameterizedGRAPEConfig(
            num_tslots=48,
            max_iter=80,
            num_restarts=3,
            seed=20260513,
            show_progress=True,
        ),
    )
    smoke_result = smoke_optimizer.optimize()
    pprint(smoke_result.to_json())
else:
    print("Smoke GRAPE is disabled. Flip RUN_SMOKE_GRAPE to True to run it interactively.")

## 6. Next reproduction steps

For a closer paper reproduction, the next useful layers are:

- Re-run the full `experiments/reproduce_evered2023_parallel_cz_gate.py` scan with a fixed choice of `two-photon`, `two-photon-detuning`, or `effective` model.
- Add the smooth-amplitude ansatz from Methods after the fixed-amplitude branch is stable.
- Add non-Hermitian or Lindblad intermediate-state scattering, Rydberg decay, dephasing, finite AOM rise time, finite blockade, Doppler/motional samples, and eventually the benchmarking/SPAM layer.
- Keep new outputs under `artifacts/evered2023_parallel_cz/` with descriptive names instead of overwriting committed artifacts casually.